# 2.1 Refactorización de P1

Lo elegido en `1.1-dm-p1-exploracion-modelos` se convierte en clases con interfaz `fit` / `predict`, igual que scikit-learn, para que el controlador común pueda entrenar cualquier pregunta de la misma forma. Al final se compara con `mibici_equipo39/modeling/p1.py`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from mibici_equipo39 import config as C
from mibici_equipo39.features import FEATURE_COLUMNS, eligible_stations, load_tables

metadata, X, y = load_tables()

## Baseline como clase

En la exploración eran dos `groupby` sueltos. Como clase guarda las medias de train en `fit` y en `predict` usa la de estación, día y hora; si esa combinación no existe, cae a estación y hora y luego a la global.

In [2]:
class SeasonalMeanBaseline:
    levels = (["station_id", "day_of_week", "hour_of_day"], ["station_id", "hour_of_day"])

    def fit(self, frame, y):
        self.means_ = [y.groupby([frame[k] for k in keys], observed=True).mean().rename("mean") for keys in self.levels]
        self.global_ = float(y.mean())
        return self

    def predict(self, frame):
        pred = pd.Series(np.nan, index=frame.index)
        for keys, means in zip(self.levels, self.means_):
            pred = pred.fillna(frame[keys].join(means, on=keys)["mean"])
        return pred.fillna(self.global_).to_numpy()

## Métricas por conjunto

Una sola función calcula `train_`, `val_` y `test_` con el prefijo del conjunto, como pide la buena práctica de MLflow.

In [3]:
SPLITS = {"train": "train", "val": "validation", "test": "test"}

def regression_metrics(y, pred, split):
    out = {}
    for prefix, name in SPLITS.items():
        mask = split.eq(name).to_numpy()
        out[f"{prefix}_mae"] = mean_absolute_error(y[mask], pred[mask])
        out[f"{prefix}_rmse"] = root_mean_squared_error(y[mask], pred[mask])
    return out

## Experimento

`P1Experiment` junta el filtro de estaciones, el entrenamiento de los dos modelos y la regla de selección. El registro en MLflow se agrega solo en el módulo.

In [4]:
class P1Experiment:
    def __init__(self, metadata, X, y):
        keep = metadata["station_id"].isin(eligible_stations(metadata, y)).to_numpy()
        self.metadata = metadata[keep].reset_index(drop=True)
        self.X = X[keep].reset_index(drop=True)
        self.y = y[keep].reset_index(drop=True)
        self.train = self.metadata["split"].eq("train").to_numpy()

    def fit(self):
        frame = pd.concat([self.metadata, self.X[["day_of_week", "hour_of_day"]]], axis=1)
        self.baseline = SeasonalMeanBaseline().fit(frame[self.train], self.y[self.train])
        self.candidate = HistGradientBoostingRegressor(**C.P1_HGB_PARAMS).fit(self.X.loc[self.train, FEATURE_COLUMNS], self.y[self.train])
        self.predictions = {
            "baseline": self.baseline.predict(frame),
            "candidato": np.clip(self.candidate.predict(self.X[FEATURE_COLUMNS]), 0, None),
        }
        self.metrics = {k: regression_metrics(self.y.to_numpy(), p, self.metadata["split"]) for k, p in self.predictions.items()}
        self.selected = "candidato" if self.metrics["candidato"]["val_mae"] < self.metrics["baseline"]["val_mae"] else "baseline"
        return self

notebook = P1Experiment(metadata, X, y).fit()
print("seleccionado:", notebook.selected)
pd.DataFrame(notebook.metrics).T[["train_mae", "val_mae", "train_rmse", "val_rmse"]].round(4)

seleccionado: candidato


,train_mae,val_mae,train_rmse,val_rmse
baseline,0.6916,0.7119,1.3751,1.4495
candidato,0.6607,0.6730,1.2245,1.2820


> **Mismos números que la exploración** en validación. Train y validación quedan cerca, así que el candidato no está sobreajustado. Las métricas de prueba se calculan aquí, pero solo se reportan desde el run de MLflow.

## Paso al módulo

| Pieza | Destino |
|---|---|
| `SeasonalMeanBaseline`, `regression_metrics`, `P1Experiment` | `mibici_equipo39/modeling/p1.py` |
| Hiperparámetros y mínimo de salidas | `config.py` |
| Parámetros, etiquetas y run anidado de MLflow | `mibici_equipo39/modeling/tracking.py` |
| Gráficas | `mibici_equipo39/plots.py` |

Se ejecuta con `python -m mibici_equipo39.modeling.p1 --executor personaN`.

In [5]:
from mibici_equipo39.modeling.p1 import P1Experiment as ModuleExperiment

module = ModuleExperiment(metadata, X, y).fit()
for name in ["baseline", "candidato"]:
    assert np.allclose(notebook.predictions[name], module.predictions[name])
print("notebook y módulo dan las mismas predicciones; seleccionado:", module.selected)

notebook y módulo dan las mismas predicciones; seleccionado: candidato
